# Normalization Batch Evaluation Confusion Table

This notebook loads JSON result files from `further_evaluation/normalization_1`, aggregates results for a selected batch size, and compares the number of predicted tables with the number of expected tables. Rows distinguish correct counts, too few tables, too many tables, and no predicted tables. Columns show whether the expected tables were already in the database.

For correct table-count guesses, each cell also reports the percentage or number of correct join-key predictions. Use the batch-size, insert-row-count, and column filters to select the data shown. The absolute numbers checkbox switches between percentages and counts.

In [3]:
import glob
import json
import os
import re

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, clear_output, display

from util.insert_parser import UnexpectedTokenException, parse_insert

BASE_DIR = os.path.join(os.getcwd(), 'further_evaluation', 'normalization_1')

ROW_NAMES = [
    'Correct number of tables',
    'Too few tables predicted',
    'Too many tables predicted',
    'No tables predicted',
]
PRIMARY_COLS = ['New tables', 'Existing tables']
SUBCOLS = ['Exact name', 'Undefined']
ROW_COUNT_OPTIONS = ['All'] + list(range(1, 6))


def find_batch_sizes():
    pattern = re.compile(r'_batch_?(\d+).json$', re.IGNORECASE)
    sizes = set()
    if not os.path.isdir(BASE_DIR):
        return []
    for root, _, files in os.walk(BASE_DIR):
        for filename in files:
            match = pattern.search(filename)
            if match:
                sizes.add(int(match.group(1)))
    return sorted(sizes)


def get_table_names(entry, key):
    names = entry.get(key)
    if isinstance(names, list):
        return names
    if names is None:
        singular_key = key[:-1] if key.endswith('s') else key
        name = entry.get(singular_key)
        return [] if name is None else [name]
    return [names]


def classify_entry(entry):
    expected_tables = get_table_names(entry, 'expected_table_names')
    predicted_tables = get_table_names(entry, 'predicted_table_names')
    database_state = entry.get('database_state') or {}
    if not isinstance(database_state, dict):
        database_state = {}

    actual_is_existing = bool(expected_tables) and all(
        table_name in database_state for table_name in expected_tables
    )
    expected_count = len(expected_tables)
    predicted_count = len(predicted_tables)
    if predicted_count == 0:
        row = 'No tables predicted'
    elif predicted_count == expected_count:
        row = 'Correct number of tables'
    elif predicted_count < expected_count:
        row = 'Too few tables predicted'
    else:
        row = 'Too many tables predicted'
    col_primary = 'Existing tables' if actual_is_existing else 'New tables'

    query = entry.get('query', entry.get('insert', '')) or ''
    insert_columns = None
    insert_row_count = None
    try:
        if isinstance(query, str) and query.strip():
            parsed_insert = parse_insert(query.lower())
            insert_columns = parsed_insert.get('columns')
            values = parsed_insert.get('values')
            insert_row_count = len(values) if values is not None else None
    except (UnexpectedTokenException, Exception):
        insert_columns = None
    column_sub = 'Exact name' if insert_columns else 'Undefined'

    predicted_join_index = entry.get('predicted_join_column_index')
    expected_join_index = entry.get('expected_join_column_index')
    if isinstance(predicted_join_index, list):
        join_key_correct = expected_join_index in predicted_join_index
    else:
        join_key_correct = predicted_join_index == expected_join_index

    return row, col_primary, column_sub, join_key_correct, insert_row_count


def load_counts_for_batch(batch_size, column_filter='All', row_count_filter='All'):
    columns = pd.MultiIndex.from_product([PRIMARY_COLS, SUBCOLS])
    counts = pd.DataFrame(0, index=ROW_NAMES, columns=columns)
    join_key_counts = pd.DataFrame(0, index=ROW_NAMES, columns=columns)
    pattern = os.path.join(BASE_DIR, '**', f'*_batch_{batch_size}.json')

    for filepath in glob.glob(pattern, recursive=True):
        try:
            with open(filepath, encoding='utf-8') as result_file:
                data = json.load(result_file)
        except (OSError, json.JSONDecodeError):
            continue
        if not isinstance(data, list):
            continue

        for entry in data:
            if 'predicted_table_names' not in entry and 'predicted_table_name' not in entry:
                continue
            row, col_primary, column_sub, join_key_correct, insert_row_count = classify_entry(entry)
            if column_filter != 'All' and column_sub != column_filter:
                continue
            if row_count_filter != 'All' and insert_row_count != row_count_filter:
                continue
            column = (col_primary, column_sub)
            counts.loc[row, column] += 1
            if join_key_correct:
                join_key_counts.loc[row, column] += 1
    return counts, join_key_counts


def render_confusion_html(df, join_key_counts, as_percent=True):
    display_df = df.copy()
    if as_percent:
        for column in display_df.columns:
            total = df[column].sum()
            display_df[column] = 0.0 if total == 0 else df[column] / total * 100

    html = [
        '<table style="border-collapse: collapse; font-family: Arial;">',
        '<thead>',
        '<tr><th style="border: 1px solid #ddd; padding: 6px;"></th>',
        f'<th colspan="{len(display_df.columns)}" style="border: 1px solid #ddd; padding: 6px; text-align: center; background: #f0f0f0;">Actual values</th></tr>',
        '<tr><th style="border: 1px solid #ddd; padding: 6px; text-align: center;">Predictions</th>',
    ]
    for primary in PRIMARY_COLS:
        html.append(
            f'<th colspan="{len(SUBCOLS)}" style="border: 1px solid #ddd; padding: 6px; text-align: center; background: #f7f7f7;">{primary}</th>'
        )
    html.append('</tr><tr><th style="border: 1px solid #ddd; padding: 6px;"></th>')
    for _, subcolumn in display_df.columns:
        html.append(
            f'<th style="border: 1px solid #ddd; padding: 6px; text-align: center;">{subcolumn}</th>'
        )
    html.append('</tr></thead><tbody>')

    for row in display_df.index:
        html.append(f'<tr><th style="border: 1px solid #ddd; padding: 6px; text-align: left;">{row}</th>')
        for primary, subcolumn in display_df.columns:
            column = (primary, subcolumn)
            value = display_df.loc[row, column]
            if as_percent:
                value_text = f'{value:.1f}%'
            else:
                value_text = str(int(df.loc[row, column]))
            if row == 'Correct number of tables':
                join_total = int(df.loc[row, column])
                join_correct = int(join_key_counts.loc[row, column])
                if as_percent:
                    join_value = 0.0 if join_total == 0 else join_correct / join_total * 100
                    value_text += f'<br><small>join key: {join_value:.1f}%</small>'
                else:
                    value_text += f'<br><small>join key: {join_correct}/{join_total}</small>'
            correct = row == 'Correct number of tables'
            background = 'lightgreen' if correct else 'salmon'
            html.append(
                f'<td style="border: 1px solid #ddd; width: 130px; padding: 8px; text-align: center; background-color: {background};">{value_text}</td>'
            )
        html.append('</tr>')
    html.extend(['</tbody>', '</table>'])
    return '\n'.join(html)

In [ ]:
batch_sizes = find_batch_sizes()
if not batch_sizes:
    print('No batch files found under', BASE_DIR)

batch_dropdown = widgets.Dropdown(
    options=batch_sizes,
    value=batch_sizes[0] if batch_sizes else None,
    description='Batch:',
)
abs_checkbox = widgets.Checkbox(value=False, description='Absolute numbers')
col_filter_dropdown = widgets.Dropdown(
    options=['All'] + SUBCOLS,
    value='All',
    description='Column filter:',
)
row_count_dropdown = widgets.Dropdown(
    options=ROW_COUNT_OPTIONS,
    value='All',
    description='Insert rows:',
)
output = widgets.Output()


def update_table():
    with output:
        clear_output(wait=True)
        if batch_dropdown.value is None:
            return
        counts, join_key_counts = load_counts_for_batch(
            batch_dropdown.value,
            column_filter=col_filter_dropdown.value,
            row_count_filter=row_count_dropdown.value,
        )
        if counts.values.sum() == 0:
            print(
                f'No entries for batch {batch_dropdown.value} '
                f'(rows={row_count_dropdown.value}, filter={col_filter_dropdown.value})'
            )
            display(counts)
            return
        display(
            HTML(
                render_confusion_html(
                    counts,
                    join_key_counts,
                    as_percent=not abs_checkbox.value,
                )
            )
        )


def on_change(change):
    if change['name'] == 'value':
        update_table()


batch_dropdown.observe(on_change)
abs_checkbox.observe(on_change)
col_filter_dropdown.observe(on_change)
row_count_dropdown.observe(on_change)
display(widgets.HBox([batch_dropdown, abs_checkbox, col_filter_dropdown, row_count_dropdown]))
display(output)
update_table()

Output()